# W01 · Implicit Neural Representations & spectral bias
# W01 · 隱式神經表示與頻譜偏差

**English.** An *implicit neural representation* (INR) stores a signal as a
function `f(coordinate) -> value` realized by a neural network, instead of an
array of pixels. A plain coordinate MLP suffers from **spectral bias**: it
learns low frequencies fast and high frequencies slowly (or never), so the
reconstruction looks blurry. This motivates positional encoding (W02) and
learned grids (W03), which PEPS unifies.

**繁體中文.** 隱式神經表示(INR)把訊號存成一個由神經網路實現的函式
`f(座標)->數值`,而非像素陣列。純座標 MLP 有**頻譜偏差**:低頻學得快、
高頻學得慢(甚至學不到),重建結果糊糊的。這正是位置編碼(W02)與可學習
grid(W03)的動機,而 PEPS 把兩者統一起來。

In [1]:
# Repo bootstrap: make `peps` and `apps` importable from the notebook.
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import torch
from peps.train import auto_device
device = auto_device()
print('torch', torch.__version__, '| device', device)

torch 2.10.0+rocm7.0 | device cuda


/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory


## 1. Load one Kodak image / 載入一張 Kodak 影像

In [2]:
from apps.image.data import load_image, image_to_coords_targets, find_kodak
img = load_image(find_kodak(1), max_size=256)   # (H, W, 3) in [0,1]
coords, targets, (H, W) = image_to_coords_targets(img)
print('image', H, W, '| coords', coords.shape, '| targets', targets.shape)
import matplotlib.pyplot as plt
plt.imshow(img); plt.title('target Kodak image'); plt.axis('off'); plt.show()

image 171 256 | coords torch.Size([43776, 2]) | targets torch.Size([43776, 3])


## 2. Fit a plain MLP (no positional encoding) / 純 MLP 擬合(無位置編碼)
Watch it converge to a blurry, low-frequency version. 觀察它收斂成糊掉的低頻版本。

In [3]:
from apps.image.build import build_plain_mlp
from peps.train import fit, TrainConfig, render_full
from peps.metrics import psnr

model, pc = build_plain_mlp(num_frequencies=0)   # raw (x,y) input
losses = []
fit(model, coords, targets,
    TrainConfig(steps=1500, batch_size=16384, lr=1e-2, device=device),
    on_log=lambda s, l: losses.append((s, l)))
pred = render_full(model, coords, device=device).reshape(H, W, 3).clamp(0, 1)
print(f'plain MLP: params={pc}  PSNR={psnr(pred, img):.2f} dB')

fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].imshow(img); ax[0].set_title('target'); ax[0].axis('off')
ax[1].imshow(pred); ax[1].set_title(f'plain MLP ({psnr(pred, img):.1f} dB)'); ax[1].axis('off')
plt.show()

plain MLP: params=8707  PSNR=18.82 dB


## 3. The spectral-bias diagnostic / 頻譜偏差診斷
Compare the FFT magnitude of target vs reconstruction; the high-frequency
ring is missing. 比較目標與重建的 FFT 幅值,高頻環會缺失。

In [4]:
import torch
def logmag(x):
    g = x.mean(-1)  # grayscale
    F = torch.fft.fftshift(torch.fft.fft2(g))
    return torch.log(F.abs() + 1e-6)
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].imshow(logmag(img), cmap='magma'); ax[0].set_title('target FFT'); ax[0].axis('off')
ax[1].imshow(logmag(pred), cmap='magma'); ax[1].set_title('plain MLP FFT'); ax[1].axis('off')
plt.show()

## 4. Takeaway / 小結
Plain coordinate MLPs cannot represent fine detail. Next week we add
**positional encoding** and view it through the Lissajous lens that PEPS uses.

純座標 MLP 無法表示細節。下週加入**位置編碼**,並用 PEPS 的 Lissajous 視角來理解它。